# GDP Extract

**Source:** World Bank indicator NY.GDP.MKTP.CD (GDP current US$)

**Output:** `data/country_gdp.csv`

In [1]:
import os
import pandas as pd
from pathlib import Path

if Path.cwd().name == "extract":
    os.chdir("..")

In [2]:
# Read raw CSV
raw_gdp = pd.read_csv(
    "raw/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_126992/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_126992.csv",
    encoding="utf-8-sig"
)
print(f"Shape: {raw_gdp.shape}")
print(raw_gdp.head())

Shape: (266, 70)
                  Country Name Country Code     Indicator Name   
0                        Aruba          ABW  GDP (current US$)  \
1  Africa Eastern and Southern          AFE  GDP (current US$)   
2                  Afghanistan          AFG  GDP (current US$)   
3   Africa Western and Central          AFW  GDP (current US$)   
4                       Angola          AGO  GDP (current US$)   

   Indicator Code          1960          1961          1962          1963   
0  NY.GDP.MKTP.CD           NaN           NaN           NaN           NaN  \
1  NY.GDP.MKTP.CD  2.420569e+10  2.495889e+10  2.707323e+10  3.176914e+10   
2  NY.GDP.MKTP.CD           NaN           NaN           NaN           NaN   
3  NY.GDP.MKTP.CD  1.190481e+10  1.270772e+10  1.363059e+10  1.446891e+10   
4  NY.GDP.MKTP.CD           NaN           NaN           NaN           NaN   

           1964          1965  ...          2016          2017          2018   
0           NaN           NaN  ...  2.98363

In [3]:
# Load ISO 3166 reference
iso_codes = pd.read_csv("data/country_name_iso_3166.csv")["alpha_3_code"].unique()

# Select 2024 data, rename columns, filter to ISO codes
gdp_2024 = raw_gdp[["Country Code", "2024"]].copy()
gdp_2024.columns = ["alpha_3_code", "gdp_usd"]
gdp_2024 = gdp_2024.dropna()
gdp_2024 = gdp_2024[gdp_2024["alpha_3_code"].isin(iso_codes)]
gdp_2024["gdp_usd"] = gdp_2024["gdp_usd"].astype("int64")
gdp_2024["year"] = 2024
gdp_2024 = gdp_2024.sort_values("alpha_3_code").reset_index(drop=True)

print(f"Rows after filtering: {len(gdp_2024)}")
print(gdp_2024.head())

Rows after filtering: 191
  alpha_3_code       gdp_usd  year
0          ABW    4265650673  2024
1          AGO  100998916781  2024
2          ALB   27046429296  2024
3          AND    4039842405  2024
4          ARE  552324846834  2024


In [4]:
# Cross-check: find ISO codes in reference but missing from 2024 data
missing = set(iso_codes) - set(gdp_2024["alpha_3_code"])
print(f"ISO codes with no 2024 GDP data ({len(missing)}): {sorted(missing)}")

ISO codes with no 2024 GDP data (58): ['AFG', 'AIA', 'ALA', 'ASM', 'ATA', 'ATF', 'BES', 'BLM', 'BTN', 'BVT', 'CCK', 'COK', 'CUB', 'CXR', 'CYM', 'ERI', 'ESH', 'FLK', 'GGY', 'GIB', 'GLP', 'GRL', 'GUF', 'GUM', 'HMD', 'IMN', 'IOT', 'JEY', 'LBN', 'LIE', 'MAF', 'MNP', 'MSR', 'MTQ', 'MYT', 'NFK', 'NIU', 'PCN', 'PLW', 'PRK', 'REU', 'SGS', 'SHN', 'SJM', 'SMR', 'SPM', 'SSD', 'SYR', 'TKL', 'TON', 'TUV', 'TWN', 'UMI', 'VAT', 'VGB', 'VIR', 'WLF', 'YEM']


In [5]:
# Save to data/country_gdp.csv
gdp_2024.to_csv("data/country_gdp.csv", index=False)
print(f"Saved {len(gdp_2024)} rows to data/country_gdp.csv")

Saved 191 rows to data/country_gdp.csv
